In [ ]:
"""
24/08/2024 (updated with SSP-specific EOL data)

authors : @ADanneaux
          @alperenyayla

This code takes SOL and EOL data and returns the list of scenarios
to be tested by Main_dLCA.py

- LCI_SOL_2025.csv is SOL-side LCI (per SOL: BAU, LC3, EST)
- LCI_EOL_2100.csv is EOL-side LCI with SSP-specific columns:
    C1_SSP1, C2_SSP1, ..., T8_SSP5

It generates:
- Scenario_LCI.csv  (for GWP runs)
- Input_AGTP.csv    (for AGTP runs)
"""

# %% Importing libraries & data
import pandas as pd
import os

raw_data_path = os.path.join("..", "..", "raw_data")
treated_data_path = os.path.join("..", "..", "generated_data")

# SOL: one column per SOL (BAU, LC3, EST)
LCI_SOL = pd.read_csv(os.path.join(raw_data_path, "LCI_SOL_2025.csv"), index_col=0)

# EOL: columns are now EOLtype_SSPlabel (e.g. C1_SSP1, C1_SSP2, C1_SSP5, ...)
LCI_EOL = pd.read_csv(os.path.join(raw_data_path, "LCI_EOL_2100.csv"), index_col=0)


# %% Default data

C_Pine_Mass = {
    "BAU": 12047.04,
    "LC3": 12047.04,
    "EST": 415416.13,
}

ConcreteVolume = {
    "BAU": 1939,
    "LC3": 1939,
    "EST": 773,
}

Area_in = {
    "BAU": 3097.5,
    "LC3": 3097.5,
    "EST": 0,
}

Cemtype = {
    "BAU": 0,
    "LC3": 1,
    "EST": 0,
}

LFD_index = {
    "BAU_C1": 2,
    "BAU_C2": 0,
    "BAU_C3": 1,
    "LC3_C1": 5,
    "LC3_C2": 3,
    "LC3_C3": 4,
}


# %% Defining the dataframe

# Building types: BAU_C1..C3, LC3_C1..C3, EST_T1..T8
building_types = [x + "_" + y for x in ["BAU", "LC3"] for y in ["C1", "C2", "C3"]] + [
    "EST_T" + str(x) for x in range(1, 9)
]


def filling_scenario(Data, scenario, Dynamic, THI, SSP_index, TexposureEOL):
    """
    Fill a single scenario column in Data.

    scenario string examples:
    - BAU_C1_Static_20_SSP1
    - LC3_C2_Dynamic_100_SSP2
    - EST_T3_0.25_SSP5   (for AGTP input)

    SOL = BAU / LC3 / EST (first token)
    EOL = C1 / C2 / C3 / T1..T8 (second token)
    SSP_label = SSP1 / SSP2 / SSP5 (last token)
    """

    parts = scenario.split("_")

    SOL = parts[0]                # BAU / LC3 / EST
    EOL = parts[1]                # C1 / C2 / C3 / T1..T8
    SSP_label = parts[-1]         # SSP1 / SSP2 / SSP5

    Data.loc["Dynamic", scenario] = int(Dynamic)
    Data.loc["THI", scenario] = int(THI)
    Data.loc["SSP", scenario] = int(SSP_index)  # 0:SSP1, 1:SSP2, 2:SSP5

    # Pine mass
    Data.loc["C_Pine_Mass", scenario] = C_Pine_Mass[SOL]

    # -------- SOL side (unchanged: one column per SOL) --------
    for index in LCI_SOL.index:
        Data.loc[index, scenario] = LCI_SOL.loc[index, SOL]

    # -------- EOL side (NEW: SSP-specific columns) --------
    # Column name looks like e.g. "C1_SSP1", "T3_SSP2", "T8_SSP5", ...
    eol_col = f"{EOL}_{SSP_label}"

    if eol_col not in LCI_EOL.columns:
        raise KeyError(
            f"EOL column '{eol_col}' not found in LCI_EOL. "
            f"Available columns: {list(LCI_EOL.columns)}"
        )

    for index in LCI_EOL.index:
        Data.loc[index, scenario] = LCI_EOL.loc[index, eol_col]

    # -------- Other building parameters --------
    Data.loc["Area_in", scenario] = Area_in[SOL]
    # All buildings studied are assumed to be fully cladded
    Data.loc["Area_out", scenario] = 0

    Data.loc["Cemtype", scenario] = Cemtype[SOL]

    # LFD_index: special case for EST, otherwise use map by SOL+EOL
    Data.loc["LFD_index", scenario] = 2 if SOL == "EST" else LFD_index[SOL + "_" + EOL]

    Data.loc["ConcreteVolume", scenario] = ConcreteVolume[SOL]

    # -------- EOL treatment shares --------
    if EOL in ["T1", "C1", "C2", "C3"]:
        # Assume 2021 treatment share of construction and demolition waste
        # (World Bank waste database, Kaza et al., 2018)
        Data.loc["landfill_Ashare", scenario] = 0.537
        Data.loc["landfill_Bshare", scenario] = 0.019
        Data.loc["landfill_Cshare", scenario] = 0.022
        Data.loc["recycling_share", scenario] = 0.238
        Data.loc["regrowth_share", scenario] = 0.000
        Data.loc["incineration_share", scenario] = 0.184

    else:
        Data.loc["landfill_Ashare", scenario] = 1 if EOL == "T6" else 0
        Data.loc["landfill_Bshare", scenario] = 1 if EOL == "T7" else 0
        Data.loc["landfill_Cshare", scenario] = 1 if EOL == "T8" else 0
        Data.loc["recycling_share", scenario] = 1 if EOL in ["T3", "T4", "T5"] else 0
        Data.loc["regrowth_share", scenario] = 1 if EOL == "T5" else 0
        Data.loc["incineration_share", scenario] = 1 if EOL == "T2" else 0

    Data.loc["TexposureEOL", scenario] = TexposureEOL

    return Data


# %% Scenario_LCI (for GWP runs)

Data = pd.DataFrame()

# SSP labels in scenario name and their index used in SSP field
ssp_suffixes = ["_SSP1", "_SSP2", "_SSP5"]  # 0: SSP1, 1: SSP2, 2: SSP5

for Dynamic, method in enumerate(["Static", "Dynamic"]):
    for ind_SSP, SSP in enumerate(ssp_suffixes):
        for THI in [20, 100, 200]:
            for building in building_types:
                scenario = building + "_" + method + "_" + str(THI) + SSP
                print(scenario)
                TexposureEOL = 0.25  # concrete exposed for 3 months after demolition
                Data = filling_scenario(Data, scenario, Dynamic, THI, ind_SSP, TexposureEOL)

directory = treated_data_path
if not os.path.exists(directory):
    os.makedirs(directory)

Data.to_csv(os.path.join(treated_data_path, "Scenario_LCI.csv"), index=True)

# %% Input_AGTP (for AGTP runs)

Data = pd.DataFrame()

for ind_SSP, SSP in enumerate(ssp_suffixes):
    for TexposureEOL in [0.25, 5, 10]:  # can extend with 5,10,... if needed
        for building in building_types:
            Dynamic = 1
            THI = 300
            scenario = building + "_" + str(TexposureEOL) + SSP
            print(scenario)
            Data = filling_scenario(Data, scenario, Dynamic, THI, ind_SSP, TexposureEOL)

Data.to_csv(os.path.join(treated_data_path, "Input_AGTP.csv"), index=True)
# %%
